# Experiment: Case-001 Mitapivat Indonesia Access Evidence Gap

Objective:
- Validate the May 24 mitapivat Indonesia access evidence-gap map.
- Keep public BPOM and trial-geography checks separate from local availability claims.
- Confirm that owner routing is label-only and not access, import, travel, dosing, or trial-screening advice.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

DATA_PATH = Path(
    "../../../data/regulatory/mitapivat/2026-05-24-mitapivat-indonesia-access-evidence-gap.json"
)
gate = json.loads(DATA_PATH.read_text(encoding="utf-8"))
gate["gate_label"]

## Evidence Gap Checks

The gate should encode absence from the current public pass as `local_access_unverified`, not as proof of non-access.


In [ ]:
assert gate["checked_date"] == "2026-05-24"
assert gate["gate_label"] == "mitapivat_indonesia_access_evidence_gap_ready"
assert "local_access_unverified" in gate["allowed_public_labels"]
assert "owner_verification_required" in gate["allowed_public_labels"]
assert "unavailable" not in gate["allowed_public_labels"]

bpom_checks = gate["bpom_public_query_checks"]
assert len(bpom_checks) == 4
assert {check["query"].lower() for check in bpom_checks} == {
    "mitapivat",
    "pyrukynd",
    "aqvesme",
}
assert all(check["records_filtered"] == 0 for check in bpom_checks)

summary = gate["clinicaltrials_query_summary"]
assert summary["records_returned"] == 6
assert summary["records_with_indonesia_location"] == 0
assert summary["direct_indonesia_query_records"] == 0

summary

## Owner And Privacy Checks

The gate should route to accountable owner categories and block public operational instructions.


In [ ]:
expected_owner_order = [
    "treating_hematologist",
    "hospital_pharmacy",
    "regulatory_affairs_or_bpom_facing_owner",
    "manufacturer_medical_information",
    "payer_or_financial_assistance_owner",
]
blocked = set(gate["blocked_public_content"])

assert gate["owner_verification_order"] == expected_owner_order
assert "access_import_or_travel_instructions" in blocked
assert "dose_or_dosing_requests" in blocked
assert "trial_screening_instruction" in blocked
assert "patient_specific_treatment_instructions" in blocked
assert "screenshots" in blocked

{
    "owner_steps": len(gate["owner_verification_order"]),
    "blocked_public_items": len(blocked),
}

## Decision

The access evidence gap is ready if it blocks unsupported availability claims and sends the work to clinician, pharmacy, regulatory, manufacturer, or payer owners.
